In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
import pandas as pd
import numpy as np
import string
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.metrics.pairwise import cosine_similarity

In [3]:
TRAIN_PATH = '/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv'
train_df = pd.read_csv(TRAIN_PATH)
print("Dataset loaded successfully. Shape:", train_df.shape)

Dataset loaded successfully. Shape: (2000, 8)


In [4]:
# Q1: Frequency Distribution & Sum of Extremes

answer_counts = train_df['answer'].value_counts()
most_freq_count = answer_counts.iloc[0]
least_freq_count = answer_counts.iloc[-1]
q1_sum = most_freq_count + least_freq_count

print("Q1: Frequency Distribution of correct answers:")
print(answer_counts)
print(f"Sum of occurrences of the most and least frequent options: {q1_sum}")

Q1: Frequency Distribution of correct answers:
answer
B    490
C    459
A    369
D    358
E    324
Name: count, dtype: int64
Sum of occurrences of the most and least frequent options: 814


In [5]:
# Q2: Cleaned Vocabulary Size

def clean_text(text):
    if not isinstance(text, str):
        return []
   
    text = text.lower()

    text = text.translate(str.maketrans('', '', string.punctuation))
 
    return text.split()

unique_words = set()
for prompt in train_df['prompt']:
    unique_words.update(clean_text(prompt))

print(f"Q2: Total number of unique words across the cleaned prompt column: {len(unique_words)}")

Q2: Total number of unique words across the cleaned prompt column: 859


In [6]:
# Q3: Filtering Stop Words for Row ID 1


row_1 = train_df[train_df['id'] == 1].iloc[0]
row_1_words = clean_text(row_1['prompt'])
filtered_row_1_words = [w for w in row_1_words if w not in ENGLISH_STOP_WORDS]

print(f"Q3: Number of words left in Row ID 1 prompt after stop word filtering: {len(filtered_row_1_words)}")

Q3: Number of words left in Row ID 1 prompt after stop word filtering: 13


In [7]:
# Q4: TF-IDF Vectorizer Vocabulary Size

combined_texts = (
    train_df['prompt'].fillna('') + " " +
    train_df['A'].fillna('') + " " +
    train_df['B'].fillna('') + " " +
    train_df['C'].fillna('') + " " +
    train_df['D'].fillna('') + " " +
    train_df['E'].fillna('')
).tolist()

vectorizer = TfidfVectorizer(stop_words='english')
vectorizer.fit(combined_texts)
feature_columns_count = len(vectorizer.get_feature_names_out())

print(f"Q4: Total number of feature columns (vocabulary size) generated: {feature_columns_count}")

Q4: Total number of feature columns (vocabulary size) generated: 2762


In [8]:
# Q5: Cosine Similarity for Row ID 1 (Prompt vs Option A)

prompt_vec_1 = vectorizer.transform([row_1['prompt']])
optA_vec_1 = vectorizer.transform([str(row_1['A'])])
similarity_score_1 = cosine_similarity(prompt_vec_1, optA_vec_1)[0][0]

print(f"Q5: Cosine similarity between prompt and Option A for Row ID 1: {similarity_score_1:.4f}")

Q5: Cosine similarity between prompt and Option A for Row ID 1: 0.2720


In [9]:
# Q6: Highest Cosine Similarity Match Percentage

correct_sim_matches = 0
options = ['A', 'B', 'C', 'D', 'E']

for idx, row in train_df.iterrows():
    p_vec = vectorizer.transform([str(row['prompt'])])
    sims = []
    
    for opt in options:
        o_vec = vectorizer.transform([str(row[opt])])
        sims.append(cosine_similarity(p_vec, o_vec)[0][0])
    
    best_opt = options[np.argmax(sims)]
    if best_opt == row['answer']:
        correct_sim_matches += 1

match_percentage = (correct_sim_matches / len(train_df)) * 100
print(f"Q6: Percentage of instances where highest cosine similarity matches the correct answer: {match_percentage:.2f}%")

Q6: Percentage of instances where highest cosine similarity matches the correct answer: 13.55%


In [10]:
def map_at_3(true_answer, predicted_list):

 
    if true_answer == predicted_list[0]:
        return 1.0
   
    elif true_answer == predicted_list[1]:
        return 0.5
  
    elif true_answer == predicted_list[2]:
        return 1/3
   
    else:
        return 0.0

In [11]:
# Question 7

q7_ground_truth = 'C'
q7_predictions = ['C', 'A', 'B']

q7_score = map_at_3(q7_ground_truth, q7_predictions)
print(f"Question 7: Ground truth '{q7_ground_truth}', Predictions {q7_predictions}")
print(f"MAP@3 Score: {q7_score:.4f}")
print("-" * 50)

Question 7: Ground truth 'C', Predictions ['C', 'A', 'B']
MAP@3 Score: 1.0000
--------------------------------------------------


In [12]:
# Question 8

q8_ground_truth = 'B'
q8_predictions = ['D', 'B', 'E']

q8_score = map_at_3(q8_ground_truth, q8_predictions)
print(f"Question 8: Ground truth '{q8_ground_truth}', Predictions {q8_predictions}")
print(f"MAP@3 Score: {q8_score:.4f}")

Question 8: Ground truth 'B', Predictions ['D', 'B', 'E']
MAP@3 Score: 0.5000


In [13]:
# Function for Q9 and Q10

def map_at_3(true_answer, predicted_list):
    """Calculates MAP@3 for a single row."""
    if true_answer == predicted_list[0]:
        return 1.0
    elif true_answer == predicted_list[1]:
        return 0.5
    elif true_answer == predicted_list[2]:
        return 1/3
    else:
        return 0.0


# Q9: Majority Class Baseline MAP@3

top_3_classes = answer_counts.index[:3].tolist()
majority_baseline_scores = []

for idx, row in train_df.iterrows():
    majority_baseline_scores.append(map_at_3(row['answer'], top_3_classes))

majority_map3 = np.mean(majority_baseline_scores)
print(f"Q9: Overall MAP@3 score of the 'Majority Class' baseline: {majority_map3:.4f}")


# Q10: TF-IDF Pipeline MAP@3

tfidf_pipeline_scores = []

for idx, row in train_df.iterrows():
    p_vec = vectorizer.transform([str(row['prompt'])])
    opt_sims = []
    
    for opt in options:
        o_vec = vectorizer.transform([str(row[opt])])
        sim = cosine_similarity(p_vec, o_vec)[0][0]
        opt_sims.append((opt, sim))
    
    opt_sims.sort(key=lambda x: x[1], reverse=True)
    top_3_preds = [x[0] for x in opt_sims[:3]]
    
    tfidf_pipeline_scores.append(map_at_3(row['answer'], top_3_preds))

tfidf_map3 = np.mean(tfidf_pipeline_scores)
print(f"Q10: Final average MAP@3 score of the TF-IDF pipeline: {tfidf_map3:.4f}")


Q9: Overall MAP@3 score of the 'Majority Class' baseline: 0.4213
Q10: Final average MAP@3 score of the TF-IDF pipeline: 0.2962
